In [1]:
import os, json
import pandas as pd
import numpy as np
from report_fct import filter_interval

def build_csv_from_summary(summary_path, data_root, output_csv="all_data.csv"):
    with open(summary_path, "r") as f:
        summary = json.load(f)

    all_rows = []
    stats = {"valid": 0, "skipped": 0}

    for run in summary:
        run_name, person, date = run["run"], run["person"], run["date"]
        run_path = os.path.join(data_root, date, person, run_name)
        
        if not os.path.isdir(run_path):
            continue

        csv_files = [f for f in os.listdir(run_path) if f.endswith(".csv")]
        if len(csv_files) != 1:
            continue

        df = pd.read_csv(os.path.join(run_path, csv_files[0]))
        intervals = run.get("intervals", [])
        
        for i, interval in enumerate(intervals):
            start_m, end_m = interval["start_time"], interval["end_time"]
            ref_start = start_m - 10
            
            # --- VERIFICATION DES CONDITIONS (OVERLAP) ---
            # 1. Disponibilité dans le fichier CSV
            if ref_start < df['SecondsSince1970'].min():
                print(f"Skipped {run_name} M-{interval['maneuver_index']}: Start too close to file beginning.")
                stats["skipped"] += 1
                continue
                
            # 2. Chevauchement avec la manœuvre précédente
            if i > 0 and ref_start < intervals[i-1]["end_time"]:
                print(f"Skipped {run_name} M-{interval['maneuver_index']}: Overlap with previous maneuver.")
                stats["skipped"] += 1
                continue

            # --- EXTRACTION ET META-DONNEES ---
            df_seg = filter_interval(df, ref_start, end_m).copy()
            
            meta = {
                "run": run_name, "rider_name": person,
                "boat_name": csv_files[0].replace(".csv", ""),
                "maneuver_type": interval["maneuver_type"],
                "interval_duration": interval["duration"],
                "start_time": start_m, "end_time": end_m,
                "target_id": interval["maneuver_index"]
            }
            
            for key, value in meta.items():
                df_seg[key] = value

            # Marquage 0 vs Index (uniquement pendant la manœuvre réelle)
            mask = (df_seg['SecondsSince1970'] >= start_m) & (df_seg['SecondsSince1970'] <= end_m)
            df_seg["maneuver_index"] = np.where(mask, interval["maneuver_index"], 0)

            all_rows.append(df_seg)
            stats["valid"] += 1

    # Finalisation
    if all_rows:
        df_global = pd.concat(all_rows, ignore_index=True).sort_values('SecondsSince1970')
        df_global.to_csv(output_csv, index=False)
        print(f"\n✅ Terminé. Valides: {stats['valid']} | Rejetées: {stats['skipped']}")
    else:
        print("❌ Aucune donnée valide extraite.")

In [2]:
build_csv_from_summary(
    summary_path="summary.json",
    # data_root="../Data_Sailnjord/Maneuvers",
    data_root = "../Data_Sailnjord/Port Camargue June 2025/Maneuvers",
    output_csv="all_data.csv"
)

Skipped 08_06_2025_Run1 M-2: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-3: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-4: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-9: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-10: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-12: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-3: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-4: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-5: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-6: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-11: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-3: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-4: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-5: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-10: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-11: Overlap with previous maneuver.
Skipped 08_06_2025_

Skipped 11_06_2025_Run5 M-3: Overlap with previous maneuver.
Skipped 11_06_2025_Run5 M-4: Overlap with previous maneuver.
Skipped 11_06_2025_Run5 M-10: Overlap with previous maneuver.
Skipped 11_06_2025_Run5 M-11: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-2: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-3: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-5: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-9: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-10: Overlap with previous maneuver.
Skipped 11_06_2025_Run2 M-3: Overlap with previous maneuver.
Skipped 11_06_2025_Run2 M-4: Overlap with previous maneuver.
Skipped 11_06_2025_Run2 M-10: Overlap with previous maneuver.
Skipped 11_06_2025_Run2 M-11: Overlap with previous maneuver.
Skipped 11_06_2025_Run3 M-2: Overlap with previous maneuver.
Skipped 11_06_2025_Run3 M-3: Overlap with previous maneuver.
Skipped 11_06_2025_Run3 M-4: Overlap with previous maneuver.
Skipped 11_06_2025_


✅ Terminé. Valides: 135 | Rejetées: 115
